In [ ]:
# %% [markdown]
# # Vanda Orchid Growth Stage Classification
# 
# This notebook explores and trains models for classifying Vanda orchid growth stages.

# %% [markdown]
# ## 1. Setup and Imports

# %%
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append('../src')

# %%
# Check GPU availability
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# %% [markdown]
# ## 2. Data Exploration

# %%
# Path to data
DATA_DIR = '../data/raw'
STAGE_LABELS = ['germination', 'vegetative', 'budding', 'pre_bloom', 'full_bloom', 'wilting', 'seed_formation']

# Count images per stage
image_counts = {}
for stage in STAGE_LABELS:
    stage_dir = os.path.join(DATA_DIR, stage)
    if os.path.exists(stage_dir):
        count = len([f for f in os.listdir(stage_dir) 
                    if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        image_counts[stage] = count
    else:
        image_counts[stage] = 0

# Display counts
print("Image counts per stage:")
for stage, count in image_counts.items():
    print(f"  {stage}: {count} images")

# Visualize distribution
plt.figure(figsize=(10, 6))
plt.bar(image_counts.keys(), image_counts.values(), color='skyblue')
plt.xlabel('Growth Stage')
plt.ylabel('Number of Images')
plt.title('Distribution of Images Across Growth Stages')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# %% [markdown]
# ## 3. Data Preprocessing

# %%
# Create data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    validation_split=0.2
)

test_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# %%
# Load data
batch_size = 32
target_size = (224, 224)

train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='categorical',
    classes=STAGE_LABELS,
    subset='training',
    shuffle=True
)

validation_generator = test_datagen.flow_from_directory(
    DATA_DIR,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='categorical',
    classes=STAGE_LABELS,
    subset='validation',
    shuffle=False
)

# %%
# Display class indices
print("Class indices:", train_generator.class_indices)

# Show sample images
def plot_sample_images(generator, num_samples=5):
    samples, labels = next(generator)
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 4))
    
    for i in range(num_samples):
        axes[i].imshow(samples[i])
        axes[i].set_title(STAGE_LABELS[np.argmax(labels[i])])
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

plot_sample_images(train_generator)

# %% [markdown]
# ## 4. Model Building

# %%
from model import create_cnn_model, compile_model, get_callbacks, get_model_summary

# %%
# Create model
model = create_cnn_model(
    input_shape=(224, 224, 3),
    num_classes=len(STAGE_LABELS),
    model_type='efficientnet'
)

# Compile model
model = compile_model(model, learning_rate=0.001)

# Show summary
print(get_model_summary(model))

# %% [markdown]
# ## 5. Training

# %%
# Training parameters
EPOCHS = 50
MODEL_SAVE_PATH = '../models/vanda_growth_model.h5'

# Callbacks
callbacks = get_callbacks(MODEL_SAVE_PATH, patience=10)

# Train
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# %% [markdown]
# ## 6. Evaluation

# %%
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_title('Model Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['accuracy'], label='Training Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_title('Model Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# %%
# Evaluate on validation set
eval_results = model.evaluate(validation_generator)
print(f"Validation Loss: {eval_results[0]:.4f}")
print(f"Validation Accuracy: {eval_results[1]:.4f}")

# %%
# Confusion Matrix
y_true = validation_generator.classes
y_pred_probs = model.predict(validation_generator)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=STAGE_LABELS, yticklabels=STAGE_LABELS)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# %%
# Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=STAGE_LABELS))

# %% [markdown]
# ## 7. Test with Single Image

# %%
from predict import GrowthStagePredictor

# %%
# Load model
predictor = GrowthStagePredictor(MODEL_SAVE_PATH)

# Test prediction
test_image_path = '../data/raw/full_bloom/sample.jpg'  # Replace with actual image path
result = predictor.predict(test_image_path)

print("Prediction Results:")
print(f"Stage: {result['stage_name']}")
print(f"Confidence: {result['confidence']*100:.1f}%")
print("\nTop Predictions:")
for i, pred in enumerate(result['top_predictions'], 1):
    print(f"  {i}. {pred['stage_name']}: {pred['confidence']*100:.1f}%")

# %% [markdown]
# ## 8. Save Model Artifacts

# %%
from utils import save_model_artifacts

# %%
# Save model and artifacts
output_dir = '../models'
config = {
    'model_type': 'efficientnet',
    'input_shape': (224, 224, 3),
    'num_classes': len(STAGE_LABELS),
    'class_names': STAGE_LABELS,
    'epochs': len(history.history['loss'])
}

save_model_artifacts(model, train_generator.class_indices, config, output_dir)
print(f"Model saved to {output_dir}")